In [35]:
from PIL import Image

In [36]:
import numpy as np

In [37]:
import os

In [38]:
import pyVCT

In [39]:
mean_area_CM = {'d20': 235, 'p21': 279}
mean_area_FB = {'d20': 110, 'p21': 164}

In [40]:
config_dict = {'d20': {"GN_CM":{"value":100.09708221851317},"GN_FB":{"value":218.03245626779216},"LMAX_CM":{"value":108.50779516511425},"LMAX_FB":{"value":118.67033303131976},"DETACH_CM":{"value":186.98262745519744},"DETACH_FB":{"value":10.38711027909488},"CELL_DAY_TYPE":{"value":"d20"},"MAX_FOCALS_CM":{"value":5},"MAX_FOCALS_FB":{"value":7},"INELASTICITY_CM":{"value":59.483030773458275},"INELASTICITY_FB":{"value":19.564530549846857},"TARGETVOLUME_CM":{"value":243.13872468086456},"TARGETVOLUME_FB":{"value":169.22461469562003}},
          'p2-1': {"GN_CM":{"value":94.88372563348662},"GN_FB":{"value":151.61955134543425},"LMAX_CM":{"value":161.3629989892174},"LMAX_FB":{"value":144.91926396906112},"DETACH_CM":{"value":263.3422639629084},"DETACH_FB":{"value":9.02546849190477},"CELL_DAY_TYPE":{"value":"p2-1"},"MAX_FOCALS_CM":{"value":5},"MAX_FOCALS_FB":{"value":4},"INELASTICITY_CM":{"value":37.39539871462557},"INELASTICITY_FB":{"value":16.736100370604213},"TARGETVOLUME_CM":{"value":281.7798191755986},"TARGETVOLUME_FB":{"value":173.03626511962955}}}

In [41]:
def image_to_map(path, resolution, VOX_SIZE, cell_type, sizeMarginX, sizeMarginY, config):
    new_dict=config[cell_type]
    conf_dict={}
    for key,value in new_dict.items():
        conf_dict[key] = value['value']
    config = conf_dict
    im = Image.open(path)
    image = np.asarray(im)
    image = image/image.max()
    scale = int(resolution/VOX_SIZE)
    indentX = int(sizeMarginX/VOX_SIZE)
    indentY = int(sizeMarginY/VOX_SIZE)
    NVX, NVY = image.shape[1]*scale, image.shape[0]*scale
    mean_area = mean_area_CM[cell_type]*image.mean()+mean_area_FB[cell_type]*(1-image.mean())
    NCX, NCY = int(NVX/np.sqrt(mean_area)), int(NVY/np.sqrt(mean_area))
    sizeX = NVX*VOX_SIZE
    sizeY = NVY*VOX_SIZE
    matrix = np.zeros((int(NVY+2*indentY),int(NVX+2*indentY)))
    for i in range(image.shape[1]):
        for j in range(image.shape[0]):
            matrix[(j*scale)+indentY:((j+1)*scale+1)+indentY,indentX+(i*scale):indentX+((i+1)*scale+1)]=image[j][i]
    return sizeX, sizeY, NCX, NCY, matrix, config
    

In [42]:
sizeX, sizeY, NCX, NCY, matrix_new, config = image_to_map('Vadim.png', 0.625, 0.0025, 'd20', 0.1, 0.1, config_dict)

In [ ]:
types, c_tags, fibers, contacts = pyVCT.py_cpmfem(NCX, NCY, 0.5, 0.0025, sizeX, sizeY, 'monolayer_without_fiber', 2001, matrix_new, **config)

In [44]:
from utils.draw import make_image

In [45]:
img = make_image(types,c_tags,fibers,contacts,0)

In [46]:
from numpngw import write_png

In [47]:
write_png("example.png", img)